# 04 · Simulación del cierre de vías
**Modelo ejecutado:** una pasada BPR sobre rutas OSRM fijas por escenario. 30 réplicas por combinación, semilla global + índice de réplica. Comparaciones con números comunes entre escenarios.

Se reutilizan los 24 puntos anclados del notebook 02: la fuente espacial es uniforme discreta dentro del pool de cada zona, no un nuevo punto continuo por vehículo. Los centros y discos son proxies del estudio, no límites administrativos oficiales. Los parámetros de capacidad, demanda y demora son supuestos no calibrados.

Cada vehículo pertenece a una cohorte de salida de 15 minutos. Su contribución se cuenta en todos los segmentos de su ruta dentro de esa cohorte, y se divide entre su duración en horas para obtener veh/h. No se propagan colas, ocupación, cruces de ventana ni spillback. La capacidad de cada arco se sortea una vez por réplica. La espera Erlang es **adicional** al costo base OSRM, que ya puede incluir penalizaciones de intersección.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / 'src'))
import numpy as np
import pandas as pd
import config as c
from IPython.display import display, Image, IFrame
pd.set_option('display.max_columns', 20)
from dataclasses import asdict
from simulacion import Motor, Parametros, cargar_cache, intervalo


In [2]:
cache=cargar_cache();motor=Motor(cache)
print('Rutas por escenario:', {k:len(v) for k,v in cache['rutas'].items()})
display(pd.DataFrame(cache['manifiesto']['validacion']).T)
display(pd.Series(asdict(Parametros()),name='Valor'))
print('Capacidades nominales supuestas (veh/h/carril):',c.CAPACIDAD_POR_CARRIL)
print('Segmentos:',len(motor.arcos),'Semáforos:',len(motor.semaforos))
print('Carriles imputados:',sum(v['carriles_imputados'] for v in cache['metadatos']['segmentos'].values()))


Rutas por escenario: {'base': 552, 'vista_hermosa': 552, 'reforma': 552}


,pares_verificados,rutas_base_afectadas,segmentos_cerrados,tolerancia_s
vista_hermosa,552.0,263.0,408.0,0.2
reforma,552.0,70.0,162.0,0.2


horizonte_h                      3.0
ventana_h                       0.25
lambda_min                     800.0
lambda_max                    2300.0
pico_h                           1.5
tau_h                           0.75
intrazonal                       0.2
pesos_origen       (0.4, 0.35, 0.25)
velocidad_mu                     0.0
velocidad_sigma                  0.2
alpha                           0.15
beta                             4.0
capacidad_cv                     0.1
capacidad_min                    0.5
capacidad_max                    1.5
erlang_k                           2
erlang_lambda               0.066667
Name: Valor, dtype: object

Capacidades nominales supuestas (veh/h/carril): {'motorway': 2000.0, 'trunk': 1800.0, 'primary': 1500.0, 'secondary': 1200.0, 'tertiary': 1000.0, 'residential': 600.0, 'unclassified': 800.0, 'service': 400.0, 'living_street': 400.0}
Segmentos: 4129 Semáforos: 95
Carriles imputados: 997


Los escenarios de cierre se validaron contra todos los pares: ausencia de segmentos bloqueados, monotonía y especificidad de los tiempos de flujo libre. `cerrar.sh` restaura un snapshot original para evitar que las velocidades de un cierre persistan en el siguiente. Los derivados guardan hashes del PBF, CSV e imagen OSRM.

Para reconstruir las corridas completas desde este notebook, active la siguiente opción. Las rutas y los metadatos están incluidos; esta etapa no necesita Docker ni internet.

In [3]:
REEJECUTAR = False
if REEJECUTAR:
    import subprocess
    subprocess.run([sys.executable,str(RAIZ/'scripts/ejecutar_experimentos.py')],check=True,cwd=RAIZ)
resultado=json.loads((c.DERIVADOS/'04_resultados.json').read_text())
filas=pd.read_csv(c.DERIVADOS/'04_replicas.csv')
vehiculos=dict(np.load(c.DERIVADOS/'04_vehiculos.npz'))
from analisis import principales
resumen=principales(resultado)
display(resumen[['escenario','metrica','delta_pct','ic_pareado','delta_libre_pct','p_sin_ruta']])


,escenario,metrica,delta_pct,ic_pareado,delta_libre_pct,p_sin_ruta
0,reforma,mediana,-0.678379,"[-0.7661938589733648, -0.5881524854342794]",0.089940,0.0
1,reforma,p95,1.864653,"[1.6821918059172851, 2.043871848317961]",0.894918,0.0
2,reforma,red_total,0.002497,"[-0.03808868128726449, 0.04237203449578258]",0.809664,0.0
3,vista_hermosa,mediana,55.402445,"[54.963789067037446, 55.820923563599216]",47.788126,0.0
4,vista_hermosa,p95,45.424479,"[44.87087683784208, 45.97495064491314]",38.503387,0.0
5,vista_hermosa,red_total,41.723445,"[41.535124631412806, 41.91859346220936]",36.024003,0.0


In [4]:
# Validación ejecutable: recomputar una réplica completa y cotejar sus vehículos.
tr=motor.trafico(c.SEMILLA)
mask=vehiculos['replica']==0
np.testing.assert_array_equal(tr['origen'],vehiculos['origen'][mask])
np.testing.assert_array_equal(tr['destino'],vehiculos['destino'][mask])
for esc in c.ESCENARIOS:
    nuevo=motor.simular(tr,esc)
    np.testing.assert_allclose(nuevo['tiempos_s'],vehiculos[esc][mask],rtol=1e-12,equal_nan=True)
print('Reproducción exacta de la réplica 0 en los tres escenarios: OK')
base=filas[(filas.fuente=='pcg64')&(filas.metodo=='polar')&(filas.demanda==1)&(filas.escenario=='base')]
print('Vehículos por réplica:',base.vehiculos.min(), 'a',base.vehiculos.max(),'; media:',base.vehiculos.mean())
print('Aceptación de adelgazamiento:',base.aceptacion_nh.mean())


Reproducción exacta de la réplica 0 en los tres escenarios: OK
Vehículos por réplica: 4956 a 5357 ; media: 5090.9
Aceptación de adelgazamiento: 0.738746720944517


**Estimador:** para cada réplica se calcula la mediana, p95 y suma de tiempos en los vehículos con ruta en ambos escenarios. El titular es `100 × (media de estadísticas cerradas / media de estadísticas base − 1)`. No es la media de los cocientes ni el cambio en la mediana de todas las réplicas concatenadas. Los intervalos de 95% se obtienen con 2.000 remuestreos de réplicas completas. La suma de red es condicional a conectividad común; se reporta por separado P(sin ruta), sin convertir viajes desconectados en tiempos cero.

La comparación de flujo libre utiliza exactamente los mismos vehículos y pesos que la simulación. El +16,97% histórico usa 552 pares equiponderados: se conserva como referencia histórica, no como contrafactual directamente comparable ni cota inferior garantizada.

In [5]:
# Criterio 5: fuente de uniformes y método de normales.
comparacion=pd.DataFrame([r for r in resultado['resumen'] if r['demanda']==1 and r['escenario']=='vista_hermosa'])
display(comparacion[['fuente','metodo','metrica','delta_pct','ic_pareado']])
# Convergencia por número de réplicas, estadística mediana.
g=filas[(filas.fuente=='pcg64')&(filas.metodo=='polar')&(filas.demanda==1)&(filas.escenario=='vista_hermosa')].sort_values('replica')
display(pd.DataFrame([dict(n=n,**intervalo(g.mediana_base.iloc[:n],g.mediana_escenario.iloc[:n])) for n in sorted(set(min(n,len(g)) for n in (5,10,20,30)))]))


,fuente,metodo,metrica,delta_pct,ic_pareado
0,lcg,polar,mediana,55.443142,"[55.071863263051775, 55.836783731620294]"
1,lcg,polar,p95,45.380698,"[44.829907090894174, 45.944969971998255]"
2,lcg,polar,red_total,41.741068,"[41.530368840933086, 41.95761976975029]"
3,pcg64,polar,mediana,55.402445,"[54.963789067037446, 55.820923563599216]"
4,pcg64,polar,p95,45.424479,"[44.87087683784208, 45.97495064491314]"
5,pcg64,polar,red_total,41.723445,"[41.535124631412806, 41.91859346220936]"
6,pcg64,rechazo,mediana,55.519535,"[55.102264096214796, 55.8928713204337]"
7,pcg64,rechazo,p95,45.407381,"[44.887962658353175, 45.95019513414305]"
8,pcg64,rechazo,red_total,41.794050,"[41.58652681669818, 42.005526728003495]"
9,randu,polar,mediana,55.304380,"[54.945496943787155, 55.68249810863734]"


,n,delta_pct,ic_pareado,ic_no_pareado,reduccion_ancho_pct,replicas_validas
0,5,55.131265,"[54.318214277239335, 55.950821496007705]","[54.181902321512034, 56.02400084380505]",11.372427,5
1,10,55.586780,"[54.93023342024988, 56.22017248039073]","[54.4032932937286, 56.74910900155528]",45.011066,10
2,20,55.519362,"[54.98893357777665, 56.04262794192383]","[54.542493580884816, 56.474820135572]",45.470171,20
3,30,55.402445,"[54.96378906703743, 55.820923563599216]","[54.44720598106716, 56.25839573463698]",52.675610,30


Los cambios entre generadores se interpretan junto al error Monte Carlo. El solapamiento de IC es orientativo, no una prueba formal de equivalencia; RANDU puede fallar en dependencia multivariada sin mover este estimador. Ningún resultado debe forzarse para favorecer la hipótesis original.